# Start here: BikePulse in Google Colab

1. Choose **File → Save a copy in Drive** to keep your own notebook.
2. Keep the default **CPU** runtime; a GPU is unnecessary.
3. Run the setup cell below, then run the remaining cells in order with Shift+Enter (or Runtime → Run all).

The setup downloads our public project code and installs only the libraries needed for analysis. The data loader downloads the real UCI dataset. Colab sessions are temporary; your Drive notebook copy preserves saved code and outputs, but files in the runtime may disappear.

This notebook forecasts historical system-wide rentals, not today's station availability. Package versions are printed for reproducibility; the fully pinned local environment is in the repository. Colab results can differ slightly across package versions.

In [ ]:
# Run this first in a fresh Colab CPU runtime.
import os
import sys
import subprocess
from pathlib import Path

os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pandas>=2.2,<4', 'numpy>=1.26,<3', 'scikit-learn>=1.4,<2', 'matplotlib>=3.8,<4'])

project = Path('/content/bikepulse-demand-forecasting') if Path('/content').exists() else Path('/tmp/bikepulse-colab')
revision = '660a74c2dbf6aa5eaef84d6b76188672700710ce'
if not project.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/HoussamAlrifaii/new_project.git', str(project)])
# Use an immutable project revision. Never reset an existing checkout or overwrite edits.
current = subprocess.check_output(['git', '-C', str(project), 'rev-parse', 'HEAD'], text=True).strip()
if current != revision:
    dirty = subprocess.check_output(['git', '-C', str(project), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Existing checkout has edits. Use a fresh runtime to preserve them.')
    subprocess.check_call(['git', '-C', str(project), 'checkout', '--detach', revision])
os.chdir(project)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
from importlib.metadata import version
print('Project revision:', revision)
print('Python:', sys.version.split()[0])
print({name: version(name) for name in ['pandas', 'numpy', 'scikit-learn', 'matplotlib']})
print('Setup complete. Continue with the analysis below.')


# BikePulse: real hourly rental demand

## tl;dr
Analyzed 17,379 real Capital Bikeshare hourly observations from 2011–2012. Gradient boosting achieved test MAE of 28.3 rentals/hour versus 69.6 for the weekly baseline, a 59.3% reduction across 2,098 eligible October–December 2012 hours. This is repeated one-hour-ahead historical evaluation with observed demand lags; it does not establish live deployment accuracy.

## Context & Methods
Forecast one hour ahead using calendar information and exact clock-hour demand lags. Train before July 2012, select on July–September, and test on October–December.

### Key Assumptions
The previous observed hour is available at each forecast origin. Missing lag observations are excluded. No future observed weather or rental components enter the model. This is historical replay, not a live forecast.

Source: [UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), DOI 10.24432/C5W894. Fanaee-T & Gama; CC BY 4.0.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd() / 'bikepulse').exists() else Path.cwd().parent
sys.path.insert(0, str(root))
import pandas as pd
import matplotlib.pyplot as plt
from bikepulse.data import load_data
from bikepulse.model import prepare, train
frame = load_data()

## Data
### Validate observations and missing hours

In [ ]:
expected = pd.date_range(frame.index.min(), frame.index.max(), freq='h')
print({'observed_hours': len(frame), 'missing_hours': len(expected.difference(frame.index)), 'duplicate_hours': int(frame.index.duplicated().sum())})
display(frame[['cnt', 'holiday', 'weathersit']].describe())
display(frame.isna().sum().to_frame('null_count'))
x, y = prepare(frame)
print('Rows retained after lag requirements:', len(x))

## Results
### Daily demand and hourly commuting patterns

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)
frame['cnt'].resample('D').sum().plot(ax=axes[0], title='Daily Capital Bikeshare rentals, 2011–2012')
axes[0].set_ylabel('Rentals per day')
profile = frame.assign(weekend=frame.index.dayofweek >= 5).groupby(['hr', 'weekend'])['cnt'].mean().unstack()
profile.columns = ['Weekday', 'Weekend']
profile.plot(ax=axes[1], title='Mean hourly rentals by day type')
axes[1].set_xlabel('Hour of day')
axes[1].set_ylabel('Mean rentals per hour')
plt.show()

### Hourly coverage and demand distributions
Absent hours are not assumed to have zero rentals. Compare coverage and distribution before interpreting unusual daily totals.

In [ ]:
coverage = frame['cnt'].resample('D').count()
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
coverage.value_counts().sort_index().plot.bar(ax=axes[0], title='Recorded hours per day')
axes[0].set_xlabel('Recorded hours'); axes[0].set_ylabel('Number of days')
frame['cnt'].plot.hist(bins=40, ax=axes[1], title='Distribution of observed hourly rentals')
axes[1].set_xlabel('Rentals per hour'); axes[1].set_ylabel('Observed hours')
plt.show()

### Month-by-hour patterns
Compare monthly hourly averages separately by year. These descriptive heatmaps use all historical data and do not select model settings.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for year, ax in zip([2011, 2012], axes):
    subset = frame.loc[frame.index.year == year]
    matrix = subset.groupby(['mnth', 'hr'])['cnt'].mean().unstack().reindex(index=range(1,13), columns=range(24))
    image = ax.imshow(matrix, aspect='auto', origin='lower', vmin=0, vmax=800, cmap='viridis')
    ax.set_title(f'Mean hourly rentals, {year}')
    ax.set_xlabel('Hour'); ax.set_ylabel('Month')
    ax.set_yticks(range(12), range(1,13))
fig.colorbar(image, ax=axes, label='Mean rentals per recorded hour')
plt.show()

### Weather associations
These are observational associations, not causal effects. Weather categories may differ in calendar composition, and sparse categories are less reliable. Future observed weather is excluded from forecasting inputs.

In [ ]:
weather = frame.groupby('weathersit')['cnt'].agg(['count','mean','median'])
display(weather)
fig, axes = plt.subplots(1, 2, figsize=(11,4), constrained_layout=True)
weather['mean'].plot.bar(ax=axes[0], title='Mean rentals by observed weather category')
axes[0].set_xlabel('Source weather category'); axes[0].set_ylabel('Mean rentals/hour')
axes[1].scatter(frame['temp'] * 41, frame['cnt'], s=3, alpha=.15)
axes[1].set_title('Temperature and recorded hourly demand')
axes[1].set_xlabel('Temperature (°C, source scaling)'); axes[1].set_ylabel('Rentals/hour')
plt.show()

### Compare models on future dates

In [ ]:
model, metrics, predictions = train(frame)
display(pd.DataFrame(metrics['validation']).T)
display(pd.DataFrame({'Selected model': metrics['test_model'], 'Weekly baseline': metrics['test_weekly_baseline']}).T)
print(metrics['rows'])
predictions.assign(hour=predictions.index.hour, absolute_error=lambda d: (d.actual-d.prediction).abs()).groupby('hour')['absolute_error'].mean().plot.bar(title='Test absolute error by hour')
plt.ylabel('MAE (rentals per hour)')
plt.show()

### Diagnose held-out errors and compare another baseline
Last-hour persistence tests whether the model adds value beyond simply repeating the most recent demand. The comparison below is diagnostic; do not tune models against this test set.

In [ ]:
import numpy as np
from bikepulse.model import score
persistence = x.loc[predictions.index, 'lag_1']
display(pd.DataFrame({'Gradient boosting': score(predictions.actual, predictions.prediction), 'Weekly baseline': score(predictions.actual, predictions.weekly_baseline), 'Last-hour persistence': score(predictions.actual, persistence)}).T)
errors = predictions.assign(error=predictions.prediction-predictions.actual)
fig, axes = plt.subplots(1, 2, figsize=(11,4), constrained_layout=True)
axes[0].scatter(errors.actual, errors.prediction, s=5, alpha=.25)
limit=max(errors.actual.max(), errors.prediction.max())
axes[0].plot([0,limit],[0,limit], color='black', linestyle='--')
axes[0].set_xlabel('Observed rentals'); axes[0].set_ylabel('Predicted rentals'); axes[0].set_title('Held-out predictions vs observations')
axes[1].scatter(errors.prediction, errors.error, s=5, alpha=.25)
axes[1].axhline(0,color='black',linestyle='--')
axes[1].set_xlabel('Predicted rentals'); axes[1].set_ylabel('Prediction minus actual'); axes[1].set_title('Held-out signed errors')
plt.show()
display(errors.assign(absolute_error=lambda d:d.error.abs()).nlargest(10,'absolute_error'))

## Takeaways
Gradient boosting outperformed Ridge on the validation period and reduced held-out MAE by 59.3% versus the weekly baseline. Missing hours and lag requirements reduced 17,379 observations to 16,897 eligible rows; exclusions affect coverage. Historical coverage and changing demand limit generalization. No live station availability or causal effects are inferred. Next: rolling-origin evaluation, peak-hour error analysis, recent data, and calibrated uncertainty.